# Package

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import importlib

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
        print(f"{package_name} is already installed.")
    except ImportError:
        print(f"{package_name} is not installed. Installing via !pip ...")
        try:
            get_ipython().system(f"pip install {package_name}")
            print(f"{package_name} installed successfully.")
        except Exception as e:
            print(f"Failed to install {package_name}. Error:\n{e}")

packages = ["pyDOE2", "diversipy", "pygmo", "optproblems", "pymoo","GPy"]

for pkg in packages:
    ensure_package(pkg)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
pyDOE2 is already installed.
diversipy is already installed.
pygmo is not installed. Installing via !pip ...
  Using cached pygmo-v2.19.0.tar.gz (3.0 MB)
ERROR: pygmo from https://files.pythonhosted.org/packages/e2/12/090ba61479f60d5177a0048736d09dc028b2d65063ed44cb952df506336f/pygmo-v2.19.0.tar.gz does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
pygmo installed successfully.
optproblems is already installed.
pymoo is already installed.
GPy is already installed.


In [2]:
import sys
sys.path.insert(1, '/content/drive/MyDrive/PhD 2025-1 Offline data-driven MOP under uncertainty/ TGPR-MO 2023')
from desdeo_emo.EAs.RVEA import RVEA
from desdeo_problem.Problem import DataProblem
from desdeo_problem.testproblems.TestProblems import test_problem_builder
from pyDOE2 import lhs
import numpy as np
from framework.treedGP_framework import run_treed_GP as treedGP
import time
import scipy.io
import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import plotly.graph_objects as go
from pymoo.problems import get_problem
from pymoo.operators.sampling.lhs import LHS
from pymoo.problems.multi.omnitest import OmniTest

# Metrics
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [3]:
# Plot for paper
def plot_exp(F, title=None):
    n_obj = F.shape[1]
    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]
        non_pareto_F = np.delete(F, front_idx, axis=0)

        fig, ax = plt.subplots(figsize=(5, 5))

        ax.scatter(non_pareto_F[:, 0], non_pareto_F[:, 1],
                   s=10, color="#87CEEB", alpha=0.7)
        ax.scatter(pareto_F[:, 0], pareto_F[:, 1],
                   s=10, color="#FF7F0E", marker='D')

        ax.set_xlim(-150, 600)
        ax.set_ylim(-100, 500)
        ax.set_xlabel("f1", fontsize=25)
        #ax.set_ylabel("f2", fontsize=25)
        ax.tick_params(direction='out', length=5, width=1, colors='black')
        ax.tick_params(labelsize=18)
        ax.legend(frameon=False, fontsize=10, loc='best')
        ax.axvline(x=0, color='gray', linestyle='--', linewidth=1)
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1)

        if title:
            ax.set_title(title, fontsize=25)

        fig.tight_layout()
        plt.savefig(f"{title}.pdf", dpi=300, bbox_inches='tight')
        plt.show()



# Plot: 2 Objs and pareto front
def plot_obj_2d(F, xlim=(0, 1), ylim=(0, 1)):
    n_obj = F.shape[1]
    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]
        non_pareto_F = np.delete(F, front_idx, axis=0)

        fig = go.Figure(
            data=go.Scatter(
                x=F[:, 0],
                y=F[:, 1],
                mode='markers',
                name='Objective Values',
                marker=dict(size=6, color='#87CEEB', opacity=0.7)
            )
        )
        fig.add_trace(go.Scatter(
            x=pareto_F[:, 0],
            y=pareto_F[:, 1],
            mode='markers',
            name='Pareto Front',
            marker=dict(size=7, color='#FF7F0E', opacity=0.9, symbol='diamond')
    ))
        fig.update_layout(
            xaxis_title='f1',
            yaxis_title='f2',
            width=600,
            height=600,
            xaxis=dict(range=list(xlim)),
            yaxis=dict(range=list(ylim))
        )
    fig.show()



# Metrics: PICP
def metrics_picp(y_true, y_lower, y_upper):
    y_true = np.asarray(y_true)
    y_lower = np.asarray(y_lower)
    y_upper = np.asarray(y_upper)

    inside = (y_true >= y_lower) & (y_true <= y_upper)
    picp_per_objective = np.mean(inside, axis=0)
    picp_mean = np.mean(picp_per_objective)

    return picp_per_objective, picp_mean

def mean_std(arr):
    return np.mean(arr), np.std(arr)

# Problem

In [4]:
# Initial settings
# Problem: DTLZ1-7, omnitest

problem_name = 'omnitest'
problem_testbench = 'dtlz'

if 'dtlz' in problem_name:
  nvars = 10
  nobjs = 2
  problem_pymoo = get_problem(problem_name, n_var=nvars, n_obj=nobjs)

elif 'omnitest' in problem_name:
  nvars = 2
  nobjs = 2
  problem_pymoo = OmniTest(n_var=nvars)

#nsamples = 11*nvars-1
nsamples = 1000


x_low = problem_pymoo.xl
x_high = problem_pymoo.xu
sampling = LHS()
x_data = sampling(problem_pymoo, nsamples, seed=42).get("X")
y_data = problem_pymoo.evaluate(x_data)


x_names = [f'x{i}' for i in range(1,nvars+1)]
y_names = [f'f{i}' for i in range(1,nobjs+1)]
row_names = ['lower_bound','upper_bound']

print('problem_name', problem_name)
print('x_data', x_data.shape)
print('y_data', y_data.shape)


# Metrics: HV
# Metrics: HV
if problem_name == 'dtlz1':
  obj_min = np.array([0,0])
  obj_max = np.array([552.30,568.36])

if problem_name == 'dtlz2':
  obj_min = np.array([0,0])
  obj_max = np.array([2.78,2.93])

if problem_name == 'dtlz3':
  obj_min = np.array([0,0])
  obj_max = np.array([1605.54,1670.48])

if problem_name == 'dtlz4':
  obj_min = np.array([0,0])
  obj_max = np.array([2.83,2.78])

if problem_name == 'dtlz5':
  obj_min = np.array([0,0])
  obj_max = np.array([2.61,2.70])

if problem_name == 'dtlz6':
  obj_min = np.array([0,0])
  obj_max = np.array([9.78,9.78])

if problem_name == 'dtlz7':
  obj_min = np.array([0,0])
  obj_max = np.array([1.10,33.43])

if problem_name == 'omnitest':
  obj_min = np.array([-2,-2])
  obj_max = np.array([2.40,2.40])

ref_point = np.array([1.1,1.1])
hv = HV(ref_point=ref_point)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

n_points = 200
if problem_name == 'dtlz5':
    X_opt = np.full((n_points, nvars), 0.5)
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz6':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, 0] = np.linspace(0, 1, n_points)
    pf = problem_pymoo.evaluate(X_opt)

elif problem_name == 'dtlz7':
    X_opt = np.zeros((n_points, nvars))
    X_opt[:, :nobjs-1] = np.linspace(0, 1, n_points).reshape(-1, 1)
    pf = problem_pymoo.evaluate(X_opt)
else:
    pf = problem_pymoo.pareto_front()
igd_plus = IGDPlus(pf)

problem_name omnitest
x_data (1000, 2)
y_data (1000, 2)

Min-Max normalization -> Min:  [-2 -2]
Min-Max normalization -> Max:  [2.4 2.4]
HV Reference points:  [1.1 1.1]


In [5]:
x_data

array([[1.47253591, 3.60173333],
       [5.07193713, 5.97454806],
       [1.53576486, 3.19300391],
       ...,
       [0.05110892, 2.51653149],
       [5.74226104, 5.96195231],
       [0.8895074 , 3.8673152 ]])

# TGPR-MO

In [6]:
problem, total_points_per_model, total_points_per_model_sequence = treedGP(x_data, y_data, x_low, x_high)

Building trees...
Building leaf node GPs...
Building finished...


In [7]:
mse_list, igd_list, hv_surrogate_list, hv_real_list,  = [], [], [], []

for seed in range(1,31):
    start_time = time.time()
    evolver_opt = RVEA(problem,
                       use_surrogates=True,
                       population_size=100,
                       total_function_evaluations=10000)
    while evolver_opt.continue_evolution():
        evolver_opt.iterate()
    end_time = time.time()

    # Results
    solution = evolver_opt.population.individuals
    obj = evolver_opt.population.objectives
    f_real = problem_pymoo.evaluate(solution, return_values_of=["F"])
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)

    # MSE
    mse = mean_squared_error(f_real, obj)
    mse_list.append(mse)
    # IGD+
    igd_plus_real = float(igd_plus(f_real))
    igd_list.append(igd_plus_real)
    # HV
    f_real_normalization = (f_real - obj_min) / (obj_max - obj_min)
    obj_normalization = (obj - obj_min) / (obj_max - obj_min)
    hv_real = float(hv.do(f_real_normalization))
    hv_surrogate = float(hv.do(obj_normalization))
    hv_real_list.append(hv_real)
    hv_surrogate_list.append(hv_surrogate)

    print(f"\nSeed {seed} | Time: {end_time - start_time:.2f}s | "
          f"MSE: {mse:.2e} | "
          f"IGD+: {igd_plus_real:.2e} | "
          f"Sur HV: {hv_surrogate:.2f} | "
          f"Real HV: {hv_real:.2f}")


Seed 1 | Time: 4.39s | MSE: 1.12e-02 | IGD+: 1.40e-02 | Sur HV: 1.17 | Real HV: 1.16

Seed 2 | Time: 4.13s | MSE: 1.27e-02 | IGD+: 1.67e-02 | Sur HV: 1.17 | Real HV: 1.16

Seed 3 | Time: 5.81s | MSE: 2.13e-03 | IGD+: 1.64e-02 | Sur HV: 1.16 | Real HV: 1.16

Seed 4 | Time: 4.30s | MSE: 5.70e-03 | IGD+: 1.59e-02 | Sur HV: 1.16 | Real HV: 1.16

Seed 5 | Time: 4.15s | MSE: 1.40e-02 | IGD+: 1.67e-02 | Sur HV: 1.17 | Real HV: 1.16

Seed 6 | Time: 5.28s | MSE: 1.41e-02 | IGD+: 1.69e-02 | Sur HV: 1.17 | Real HV: 1.16

Seed 7 | Time: 4.51s | MSE: 2.48e-03 | IGD+: 1.56e-02 | Sur HV: 1.16 | Real HV: 1.16

Seed 8 | Time: 4.23s | MSE: 1.18e-02 | IGD+: 1.66e-02 | Sur HV: 1.17 | Real HV: 1.16

Seed 9 | Time: 5.59s | MSE: 2.46e-03 | IGD+: 1.50e-02 | Sur HV: 1.16 | Real HV: 1.16

Seed 10 | Time: 4.16s | MSE: 1.39e-02 | IGD+: 1.63e-02 | Sur HV: 1.17 | Real HV: 1.16

Seed 11 | Time: 4.90s | MSE: 2.74e-03 | IGD+: 1.60e-02 | Sur HV: 1.16 | Real HV: 1.16

Seed 12 | Time: 4.97s | MSE: 1.47e-02 | IGD+: 1.41e

Indicator results

In [8]:
mean_mse, std_mse = mean_std(mse_list)
mean_igd, std_igd = mean_std(igd_list)
mean_hv_real, std_hv_real = mean_std(hv_real_list)
mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

print('Problem name: ', problem_name)
print("\n=== TGPR-MO: Statistics over 30 runs ===")

print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")

Problem name:  omnitest

=== TGPR-MO: Statistics over 30 runs ===
MSE: Mean = 1.16e-02, Std = 4.71e-03
IGD+: Mean = 1.61e-02, Std = 9.97e-04
Sur HV: Mean = 1.17, Std = 0.00
Real HV: Mean = 1.16, Std = 0.00


In [9]:
print(problem_name+'_MSE_TGPR-MO'+' =',mse_list)
print(problem_name+'_IGD+_TGPR-MO'+' =',igd_list)
print(problem_name+'_HV_TGPR-MO'+' =',hv_real_list)

omnitest_MSE_TGPR-MO = [0.011237049094342196, 0.012747888860375242, 0.0021317820758992036, 0.005697760963380369, 0.013973283076783341, 0.014074190857292966, 0.0024808789620906005, 0.011800616166585082, 0.002458744625339693, 0.01389066755371986, 0.0027358547417398594, 0.014729740535414015, 0.011682047974161473, 0.013894232209204683, 0.013317787314808616, 0.013678118624755983, 0.021361209662365575, 0.01251378196869895, 0.01486310404012513, 0.014330581782579042, 0.013199254333803884, 0.013015738101792759, 0.012085238969023067, 0.015775033478204976, 0.01403640881773773, 0.013477417656084236, 0.011761231518308329, 0.013233430199113308, 0.015664388989726385, 0.0021860349192892653]
omnitest_IGD+_TGPR-MO = [0.014039319587153577, 0.016729973099947098, 0.016366058848196672, 0.01586172641305353, 0.016699572644138898, 0.016870740845973075, 0.015618443568684266, 0.01662115316481598, 0.01497609548044326, 0.01625809756752248, 0.015973518340320448, 0.014098395602563363, 0.016664355492450787, 0.0170201

In [10]:
plot_obj_2d(obj, xlim=(-1, 1), ylim=(-1, 1))
plot_obj_2d(f_real, xlim=(-100, 600), ylim=(-100, 600))
plot_obj_2d(f_real_normalization, xlim=(-0.1, 1), ylim=(-0.1, 1))